[Lab README](README.md)

# Lab 5.2: Tear down the deployed agent

Deletes the AWS resources that `5.1_agentcore_deploy.ipynb` created. Run it at the end of every rehearsal and at the end of the workshop. Lab 5 is the only lab that creates billable infrastructure, and nothing else in the workshop deletes it for you.

## How this notebook decides what to delete

Deletion is scoped **by resource tag**, never by name prefix:

    WorkshopResource=stop-ai-agent-hallucinations

A resource is deleted only if it carries that tag. Anything else in your account is left alone,
including resources whose names look like the workshop's.

This matters. An earlier version of this teardown matched IAM roles with
`role["RoleName"].startswith("AmazonBedrockAgentCoreSDKCodeBuild")` across the whole account. IAM is
global, so it reached every region and deleted five roles that this workshop never created. Name
prefixes describe what a resource is *called*; tags record who *owns* it. Only the second one is
safe to delete on.

## Two consequences worth understanding

1. **A resource that exists under a workshop name but carries no workshop tag is not deleted.** It is
   reported as `UNTAGGED_BLOCKED` and this notebook fails. That is deliberate: the resource is either
   someone else's, or it came from a deployment that forgot to tag, and neither is something a script
   should decide to delete for you.
2. **Failures are loud.** Nothing here swallows an exception. The previous version hid a `KeyError`
   behind a bare `except` and printed a clean bill of health while leaking a billable AgentCore
   Memory resource.

Run the dry run first, read the plan, then execute.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# At an AWS event: dependencies are pre-installed. Run this cell as-is.
# Self-paced (outside an AWS event): uncomment the line below first.
# ─────────────────────────────────────────────────────────────────────
# !pip install -r requirements.txt

print("Environment ready")

In [ ]:
import sys
from pathlib import Path

# Locate workshop_cleanup.py whether this notebook is run from
# 05-agentcore-deploy/ or from the repo root. Fails loudly rather than
# importing something unexpected.
candidates = [
    Path.cwd(),
    Path.cwd() / "05-agentcore-deploy",
    Path.cwd().parent / "05-agentcore-deploy",
]
module_dir = next((p for p in candidates if (p / "workshop_cleanup.py").is_file()), None)
if module_dir is None:
    raise FileNotFoundError(
        "workshop_cleanup.py not found. Run this notebook from "
        "05-agentcore-deploy/ or the repo root. "
        f"Looked in: {[str(p) for p in candidates]}"
    )
sys.path.insert(0, str(module_dir))

import workshop_cleanup as wc

clients = wc.Clients.build()

print(f"module:   {module_dir}")
print(f"region:   {wc.REGION}")
print(f"tag gate: {wc.WORKSHOP_TAG_KEY}={wc.WORKSHOP_TAG_VALUE}")

---
## Step 1: Dry run

Read-only. Lists every candidate, whether it is selected, and why. Nothing is deleted.

In [ ]:
plan = wc.build_plan(clients)
wc.print_plan(plan, dry_run=True, region=clients.region)

---
## Step 2: Review the plan

Before running the next cell, confirm that every line marked `DELETE` is a resource
`5.1_agentcore_deploy.ipynb` created.

The plan prints one line per resource that actually exists, then a single `SKIP` line counting the
names that were searched for and not found. Most of those names are from earlier versions of this
workshop, kept so a participant returning with older resources can still clean them up. On a normal
account roughly twenty of them are absent, and printing each one buried the handful of lines that
matter. Nothing is hidden by the grouping: absent means there is nothing to act on.

The lines to read closely are any marked `UNTAGGED_BLOCKED`. Those exist under a workshop name
without the workshop tag, so this notebook refuses to delete them and will fail at Step 3.

In [ ]:
for candidate in plan:
    if candidate.will_delete:
        print(candidate.describe())

---
## Step 3: Execute

Deletes the selected resources. Raises if anything failed or if any untagged workshop resource was
found, so a failed teardown cannot be mistaken for a successful one.

In [ ]:
exit_code = wc.run(clients, dry_run=False)

if exit_code != 0:
    raise RuntimeError(
        "Cleanup did not complete. Read the BLOCKED and FAILURES output above, "
        "some resources may still exist and may still be billing."
    )

print("Teardown completed with no failures.")

---
## Step 4: Verify by listing, not by trusting this notebook

A success message is a claim, not evidence. Re-running the plan should now show every workshop
resource as `not found`.

In [ ]:
after = wc.build_plan(clients)
remaining = [c for c in after if c.selection is not wc.Selection.ABSENT]

wc.print_plan(after, dry_run=True, region=clients.region)

if remaining:
    raise RuntimeError(f"{len(remaining)} workshop resource(s) still present after cleanup")

print("Verified: no workshop resources remain.")

---
## What this does NOT delete

- **Whatever `provision_agentcore.py` created.** The Gateway, the reservation
  Lambda, the Neo4j command secret, and the three `demo06-*` IAM roles carry a
  different owner tag, `demo06-agentcore=true`, because a different script owns
  them. That script tears them down, from the repository root:

      uv run setup/provision_agentcore.py status
      uv run setup/provision_agentcore.py teardown

  Run it after this notebook. Either one alone leaves the other half billing.
- **The CodeBuild source bucket.** The starter toolkit zips this folder and
  uploads it to an S3 bucket named
  `bedrock-agentcore-codebuild-sources-<account-id>-<region>` so CodeBuild has
  something to build from. The toolkit creates the bucket, does not tag it, and
  shares it across every AgentCore deployment in that account and region, not
  just this workshop's. So it falls outside the tag gate by design, and this
  teardown deliberately has no S3 deleter: deleting an untagged resource on the
  strength of its name is exactly the mistake that cost five unrelated IAM roles.
  Untagged resources are reported, never guessed at.

  It is close to free and it mostly empties itself. The toolkit puts a lifecycle
  rule on the bucket at creation time that expires objects after 7 days, and the
  uploaded source is a few megabytes, so the standing cost is a fraction of a
  cent per month and an empty bucket costs nothing at all. Remove it by hand if
  you want the account clean, once you are sure no other AgentCore deployment in
  the region is using it:

      aws s3 rm s3://bedrock-agentcore-codebuild-sources-<account-id>-<region> --recursive
      aws s3 rb s3://bedrock-agentcore-codebuild-sources-<account-id>-<region>

- **Neo4j infrastructure.** The Code Editor EC2 instance or the Central Neo4j ECS stack. Those come
  from CloudFormation: delete via AWS Console → CloudFormation → Delete Stack.
- **Your Aura instance and the graph Lab 1 built.** Neo4j Aura is not an AWS resource in your
  account, so nothing here can reach it. Delete it from the Aura console when you are done.
- **CloudWatch log groups.** Retained so you can review the run. Delete them by hand if you want.
- **Anything untagged.** By design. See the note at the top.

`CLEANUP.md` is the full reference for all of this, including the manual checks worth running in
the console afterwards.

## Command line equivalent

    python workshop_cleanup.py             # dry run by default: nothing is deleted
    python workshop_cleanup.py --dry-run   # the same read-only run, made explicit
    python workshop_cleanup.py --yes       # execute it and delete the tagged resources

The default is a dry run. Deletion happens only when you pass `--yes`. All three exit non-zero if the
teardown is incomplete, so they are safe to use in a script.

---
## You are done

Step 4 verified it by listing rather than by trusting a success message, so the
half of Lab 5 this notebook owns is gone. Run the provisioning teardown next and
Lab 5 costs you nothing further:

    uv run setup/provision_agentcore.py teardown

That leaves the CodeBuild source bucket described above, which is close to free
and empties itself, and your CloudWatch log groups, kept on purpose so you can
still read what the deployed agent did.

## Where the workshop leaves you

The agent is the same object throughout. Lab 3 built it, Lab 4 gave it a write
that a rule in the graph could refuse, Lab 5 put it behind an ARN, and none of
that changed the retrieval tool: it still imports `search_hotel_knowledge` from
the same `workshop.hybrid_retrieval` Lab 2 wrote. Deployment moved where the
agent runs and left what it does alone, which is the claim the lab was built to
test.

Two things outlive the teardown. The graph in Aura is still there, with the
`ReservationRequest` nodes the smoke tests wrote and the `max_guests` rule that
rejected one of them. And the `AGENT_RUNTIME_ARN` line in your repository-root
`.env` now names a Runtime that no longer exists; re-running
`5.1_agentcore_deploy.ipynb` overwrites it with a live one.

## Next

- **[`06-memory/`](../06-memory/README.md)** is optional, and it is the one lab
  that adds state: graph-native agent memory, with provenance on every remembered
  fact and isolation between actors. It runs locally and creates no AWS
  resources.
- **[Lab README](README.md)** and **[`CLEANUP.md`](CLEANUP.md)** cover the manual
  console checks worth doing if you are handing this account back to someone.